# Eurasian Otter Individual Identification
This notebook demonstrates how to run the FIT individual ID workflow on the Eurasian otter dataset using the provided pipeline functions.

## 1. Load data
We read the prediction CSV which already contains morphometric measurements and the output of a pre-trained sex-classification model.

In [ ]:

from pathlib import Path
import pandas as pd
from FIT_python.data_split_and_summary.data_import_utils import get_feature_cols

# path to predictions with sex probabilities
pred_path = Path('results/eurasian_otter_random_search_standard_metrics/eurasian_otter_all_predictions.csv')

df = pd.read_csv(pred_path)
# convert column names to a clean format expected by the utilities
df.columns = df.columns.str.replace('.', '_').str.replace('-', '_')
print('rows:', len(df))

# determine morphometric feature columns
feature_cols = get_feature_cols(df)
print('n_features:', len(feature_cols))


## 2. Prepare sex-model predictions
We select the probability columns of the best accuracy model and pass them to the evaluation helper.

In [ ]:

sex_cols = ['pred_best_accuracy_test_proba_f', 'pred_best_accuracy_test_proba_m']
sex_preds = (
    df[['id'] + sex_cols]
    .rename(columns={
        'pred_best_accuracy_test_proba_f': 'proba_f',
        'pred_best_accuracy_test_proba_m': 'proba_m',
    })
    .set_index('id')
)
sex_preds.head()


## 3. Sequential holdout evaluation
The `sequential_holdout.run` function generates pairwise trail comparisons, processes them with the pairwise pipeline and records metrics for each split.

In [ ]:

from FIT_python.pipeline_individual_id import sequential_holdout

summary = sequential_holdout.run(
    df,
    feature_cols,
    sex_predictions=sex_preds,
    iterations=1,
    n_jobs=1,
)
summary


## 4. Inspect one split
We load the predictions of the first split, compute the confusion matrix and plot a dendrogram to estimate the number of individuals present.

In [ ]:

from FIT_python.pipeline_individual_id.evaluation import compute_confusion, compute_bcr
from FIT_python.pipeline_individual_id.population_estimation import optimal_cutoff
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

res_path = Path('results/data/individual_id/split_0.csv')
if res_path.exists():
    df_res = pd.read_csv(res_path)
    cm = compute_confusion(df_res)
    bcr = compute_bcr(cm)
    print(cm)
    print('BCR:', bcr)

    trails = sorted(set(df_res['trail_a_id']) | set(df_res['trail_b_id']))
    dist_mat = pd.DataFrame(np.nan, index=trails, columns=trails)
    for a, b, val in zip(df_res['trail_a_id'], df_res['trail_b_id'], df_res['dist_euclidean']):
        dist_mat.at[a, b] = val
        dist_mat.at[b, a] = val
    np.fill_diagonal(dist_mat.values, 0.0)
    max_d = np.nanmax(dist_mat.values)
    dist_mat = dist_mat.fillna(max_d)

    Z = linkage(squareform(dist_mat.values), method='ward')
    plt.figure(figsize=(6,2))
    dendrogram(Z, labels=trails, orientation='top')
    plt.show()

    cutoff, _ = optimal_cutoff(dist_mat, len(trails))
    print('Ward cutoff:', cutoff)
else:
    print('split_0.csv not found -- run the evaluation first.')
